In [ ]:
# Install the required music processing and deep learning libraries
!pip install music21 tensorflow

In [ ]:
import glob
import numpy as np
import os
from music21 import converter, instrument, note, chord, stream
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Activation
from tensorflow.keras.utils import to_categorical

def get_notes(directory="/content/midi_songs/*.mid"):
    """Reads MIDI files and extracts notes and chords into a list."""
    notes = []
    # Find all midi files in the specified directory
    files = glob.glob(directory)
    if len(files) == 0:
        raise ValueError(f"No MIDI files found in {directory}. Did you upload them?")

    for file in files:
        print(f"Parsing {file}")
        midi = converter.parse(file)

        notes_to_parse = None
        try: # File has instrument parts
            s2 = instrument.partitionByInstrument(midi)
            notes_to_parse = s2.parts[0].recurse()
        except: # File has notes in a flat structure
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))

    return notes

def prepare_sequences(notes, n_vocab, sequence_length=100):
    """Maps notes to integers and creates input/output sequences for the LSTM."""
    pitches = sorted(set(item for item in notes))
    note_to_int = dict((note, number) for number, note in enumerate(pitches))

    network_input = []
    network_output = []

    for i in range(0, len(notes) - sequence_length, 1):
        sequence_in = notes[i:i + sequence_length]
        sequence_out = notes[i + sequence_length]
        network_input.append([note_to_int[char] for char in sequence_in])
        network_output.append(note_to_int[sequence_out])

    n_patterns = len(network_input)
    network_input = np.reshape(network_input, (n_patterns, sequence_length, 1))
    network_input = network_input / float(n_vocab)
    network_output = to_categorical(network_output, num_classes=n_vocab)

    return network_input, network_output, note_to_int

def create_network(network_input, n_vocab):
    """Creates the structure of the neural network."""
    model = Sequential()
    model.add(LSTM(
        256,
        input_shape=(network_input.shape[1], network_input.shape[2]),
        return_sequences=True
    ))
    model.add(Dropout(0.3))
    model.add(LSTM(256))
    model.add(Dense(128))
    model.add(Dropout(0.3))
    model.add(Dense(n_vocab))
    model.add(Activation('softmax'))

    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

def generate_notes(model, network_input, pitchnames, n_vocab, num_notes=500):
    """Generate notes from the neural network based on a sequence of notes."""
    start = np.random.randint(0, len(network_input)-1)
    int_to_note = dict((number, note) for number, note in enumerate(pitchnames))

    pattern = network_input[start]
    prediction_output = []

    for note_index in range(num_notes):
        prediction_input = np.reshape(pattern, (1, len(pattern), 1))
        prediction_input = prediction_input / float(n_vocab)
        prediction = model.predict(prediction_input, verbose=0)

        index = np.argmax(prediction)
        result = int_to_note[index]
        prediction_output.append(result)

        pattern = np.append(pattern, index)
        pattern = pattern[1:len(pattern)]

    return prediction_output

def create_midi(prediction_output, filename="/content/ai_generated_music.mid"):
    """Converts the output from the prediction to notes and creates a midi file."""
    offset = 0
    output_notes = []

    for pattern in prediction_output:
        if ('.' in pattern) or pattern.isdigit():
            notes_in_chord = pattern.split('.')
            notes = []
            for current_note in notes_in_chord:
                new_note = note.Note(int(current_note))
                new_note.storedInstrument = instrument.Piano()
                notes.append(new_note)
            new_chord = chord.Chord(notes)
            new_chord.offset = offset
            output_notes.append(new_chord)
        else:
            new_note = note.Note(pattern)
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        offset += 0.5

    midi_stream = stream.Stream(output_notes)
    midi_stream.write('midi', fp=filename)
    print(f"MIDI file saved to: {filename}")

In [ ]:
import zipfile
import os

# 1. Collect Data
print("Collecting and parsing MIDI files...")

zip_path = "/content/archive.zip"

# Check if the file exists before attempting to open it
if not os.path.exists(zip_path):
    print(f"Error: '{zip_path}' not found. Please upload your zip file to the /content directory.")
else:
    # Extract the zip file
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/midi_songs")

    # Find where the midi files actually are (in case of nested folders)
    search_path = "/content/midi_songs/**/*.mid"
    notes = get_notes(search_path)

    if not notes:
        print("Warning: No notes were parsed. Check if the files are valid MIDI format.")
    else:
        n_vocab = len(set(notes))
        pitchnames = sorted(set(item for item in notes))

        # 2. Preprocess Data
        print("Preparing sequences...")
        network_input_raw, network_output, note_to_int = prepare_sequences(notes, n_vocab)

        # 3. Build Model
        print("Building LSTM model...")
        model = create_network(network_input_raw, n_vocab)

        # 4. Train Model
        print("Training model... (This will take a few minutes)")
        model.fit(network_input_raw, network_output, epochs=8, batch_size=64)

        # 5. Generate and Convert to MIDI
        print("Generating new music sequences...")
        seed_input = [[note_to_int[n] for n in notes[i:i+100]] for i in range(0, len(notes)-100)]
        prediction_output = generate_notes(model, seed_input, pitchnames, n_vocab, num_notes=200)

        print("Saving to MIDI...")
        create_midi(prediction_output, filename="/content/ai_generated_music.mid")

        print("Done!")

Parsing /content/midi_songs/schumann/scn16_4.mid


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2007 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(


Parsing /content/midi_songs/schumann/scn16_5.mid
Parsing /content/midi_songs/schumann/scn15_2.mid


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 1997 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Schumann:   Fr\xf6hlicher Landmann, von der Arbeit zur\xfcckkehrend'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'aus Album f\xfcr die Jugend Opus 68, Nr. 10'>; getting generic Instrument
  warnings.warn(


Parsing /content/midi_songs/schumann/scn68_10.mid


...

Parsing /content/midi_songs/bach/bach_847.mid


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=7, data=b'Copyright 2004 by Bernd Kr\xfcger.'>; getting generic Instrument
  warnings.warn(


Preparing sequences...
Building LSTM model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training model... (This will take a few minutes)
Epoch 1/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - loss: 4.7452
Epoch 2/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 4.4269
Epoch 3/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.2989
Epoch 4/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.2249
Epoch 5/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.1573
Epoch 6/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 4.0738
Epoch 7/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 3.9857
Epoch 8/8
74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 3.9632
Generating new music sequences...
Saving to MIDI...
MIDI file saved to: /content/ai_generated_music.mid
Done!
